### Projektziel (Kurzbeschreibung)

In diesem Projekt wird ein Marketing-Kampagnendatensatz analysiert. Ziel ist es, (1) das Antwortverhalten auf Kampagnen zu verstehen und (2) Kundensegmente zu identifizieren, um daraus konkrete Marketing-Empfehlungen abzuleiten. Dazu werden Daten bereinigt, explorativ analysiert, Features abgeleitet, ein Klassifikationsmodell für `Antwort_Letzte_Kampagne` trainiert und eine Clusteranalyse durchgeführt.

#  Bibliotheken laden

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

from sklearn.preprocessing import StandardScaler, MinMaxScaler,OneHotEncoder, PolynomialFeatures

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression , Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings("ignore")

# 1. Datensatz laden

In [ ]:
# Datensatz laden Überblick
df = pd.read_csv('Marktkampagne.csv')
df.head(10)

# 2. Exploration und Reinigung der Daten

In diesem Abschnitt bereiten wir den Datensatz so auf, dass er für Analysen und Modelle zuverlässig nutzbar ist.  
Dazu prüfen wir die Datenstruktur, bereinigen fehlende Werte  
und vermeiden typische Fehlerquellen wie doppelte Zeilen oder falsche Datentypen.

Das Ziel ist ein konsistenter Datensatz, der sich ohne Überraschungen für EDA,  
Feature Engineering und Machine Learning verwenden lässt.  


### Überprüfung der Spaltentypen

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df['Datum_Kunde'] = pd.to_datetime(df['Datum_Kunde'],format="%d-%m-%Y")


Wir haben alle Spalten mit dem Datentyp `int` überprüft, um festzustellen, ob sie binär sind:  
`Beschwerde`, `Kampagne_1_Akzeptiert`, `Kampagne_2_Akzeptiert`, `Kampagne_3_Akzeptiert`,  
`Kampagne_4_Akzeptiert`, `Kampagne_5_Akzeptiert` und `Antwort_Letzte_Kampagne`.

Dabei konnten wir bestätigen, dass alle relevanten Spalten tatsächlich binär sind und ausschließlich die Werte **0** und **1** enthalten.


### Fehlende Werte identifizieren

In [ ]:
# Anzahl fehlender Werte pro Spalte
df.isna().sum().sort_values(ascending=False)

### Fehlende Werte behandeln
Es gibt **24 fehlende Werte** in der Variable *Einkommen*.

Da die Datensätze  ** 2240 Kunden** enthalten, ist der Anteil fehlender Werte **sehr gering**.

➡️ Kein ausreichender Grund, Zeilen zu löschen  
➡️ *Einkommen* ist eine zentrale Variable und zu wichtig, um sie zu verwerfe.

✔️ Fehlende Werte werden mittels Median-Imputation ersetzt (Best Practice)  oder mit eine Model vorhersagen. 

✔️ Keine Verzerrung der Verteilung  
✔️ Gewährleistung der Modellstabilität.

wir speichern die Fehlende werte in einer Dataframe und dann konnen wie entscheiden


In [ ]:
df_einkommen_fehlt = df[df['Einkommen'].isna()]

In [ ]:
df_einkommen_fehlt.head()
df_einkommen_fehlt.shape


In [ ]:
df = df.dropna(subset=['Einkommen']).copy()


### Ausreißer behandeln

In [ ]:
df.describe()

In [ ]:
#extremen Werte von Einkommen  666 666 auf den Median setzen. 
df.loc[df['Einkommen'] == 666666, 'Einkommen'] = df['Einkommen'].median()

sehr alte Geburtsjahre im Datensatz: 1903, 1909, 1910. 😅 Das ergibt Kunden, die über 110 Jahre alt wären – sehr wahrscheinlich Datenfehler

In [ ]:
median_geburtsjahr = df.loc[df['Geburtsjahr'] > 1920, 'Geburtsjahr'].median()
# Extremwerte ersetzen
df.loc[df['Geburtsjahr'] < 1920, 'Geburtsjahr'] = median_geburtsjahr


In [ ]:
import datetime as dt

current_year = dt.datetime.now().year
df['Alter'] = current_year - df['Geburtsjahr']

In [ ]:
df_einkommen_fehlt['Alter'] = current_year - df_einkommen_fehlt['Geburtsjahr']

In [ ]:
df['Familienstand'].unique()

Problematische / inkonsistente Werte

Allein → semantisch gleich wie „Ledig“

Absurd → offensichtlich Fehleingabe

Man lebt nur einmal → keine valide Kategorie

In [ ]:
familienstand_mapping = {
    'Allein': 'Ledig',
    'Absurd': 'Unbekannt',
    'Man lebt nur einmal': 'Unbekannt'
}
df['Familienstand'] = df['Familienstand'].replace(familienstand_mapping)
df['Familienstand'].value_counts()


In [ ]:
df[df['Familienstand'] == 'Unbekannt']

Wir prüfen fehlende Werte (Missing Values) und Duplikate.  
Fehlende Einkommen werden robust mit dem Median ersetzt, da Einkommen typischerweise  
stark schief verteilt ist und der Median weniger ausreißeranfällig als der Mittelwert ist.  

Duplikate werden entfernt, damit Kennzahlen und Modelle nicht verzerrt werden.  


### Duplikate sehen ohne ID
Inhaltlich gleiche Kunden mit unterschiedlicher ID

In [ ]:
df.duplicated(subset=df.columns.difference(['ID'])).sum()

In [ ]:
duplikat_zeilen = df[df.duplicated(subset=df.columns.difference(['ID']), keep=False)]
duplikat_zeilen


In [ ]:
duplikat_zeilen.sort_values(by=['Geburtsjahr', 'Einkommen'])


In [ ]:
duplikate_sortiert = df[
    df.duplicated(
        subset=['Geburtsjahr', 'Datum_Kunde', 'Einkommen','Familienstand','Bildungsniveau'],
        keep=False
    )
].sort_values(by=['Geburtsjahr', 'Datum_Kunde', 'Einkommen'])

duplikate_sortiert



In [ ]:
df.loc[
    df.duplicated(
        subset=['Geburtsjahr', 'Datum_Kunde', 'Einkommen'],
        keep=False
    )
].groupby(['Geburtsjahr', 'Datum_Kunde', 'Einkommen']).size().value_counts()


### Gleiche Person mehrfach importiert
→ sollte bereinigt werden

In [ ]:
#Duplikat entfernen
df = df.drop_duplicates(
    subset=['Geburtsjahr', 'Datum_Kunde', 'Einkommen'],
    keep='first'
)


In [ ]:
# Prüfen nach Geburtsjahr + Datum_Kunde + Einkommen
df.duplicated(subset=['Geburtsjahr', 'Datum_Kunde', 'Einkommen']).sum()


In [ ]:
df = df.reset_index(drop=True)

In [ ]:
df.info()

In [ ]:
df[df['Familienstand'] == 'Unbekannt']

In [ ]:
# 'Unbekannt' durch 'Ledig' ersetzen
df['Familienstand'] = df['Familienstand'].replace('Unbekannt', 'Ledig')


In [ ]:
df['Familienstand'].value_counts()


# 3. Scatterplots and visual exploration

 Korrelationsmatrix  ohne Binäre Spalten und ID :Saubere Analyse ohne Verzerrung durch 0/1-Spalten

Leichter zu erkennen, welche Features redundant oder stark zusammenhängend sind

Hilft beim Feature Engineering & Feature Selection

In [ ]:
# Alle numerischen Spalten auswählen
numerische_spalten = df.select_dtypes(include=['int64', 'float64']).columns

# Binäre Spalten ausschließen
binäre_spalten = [
    'Kampagne_1_Akzeptiert', 'Kampagne_2_Akzeptiert', 'Kampagne_3_Akzeptiert',
    'Kampagne_4_Akzeptiert', 'Kampagne_5_Akzeptiert', 'Antwort_Letzte_Kampagne',
    'Beschwerde','ID'  
]
kontinuierliche_spalten = [col for col in numerische_spalten if col not in binäre_spalten]

# Korrelationsmatrix nur für kontinuierliche Features
korrelation = df[kontinuierliche_spalten].corr()

# Heatmap

plt.figure(figsize=(12,10))
sns.heatmap(korrelation, annot=True, fmt=".2f", cmap='coolwarm')
plt.title("Korrelationsmatrix der kontinuierlichen numerischen Features")
plt.show()

In [ ]:
df_scatter = df [['Einkommen','Bildungsniveau','Familienstand',
       'Letzter_Kauf_Tage',  'Ausgaben_Wein', 'Ausgaben_Obst',
       'Ausgaben_Fleisch', 'Ausgaben_Fisch', 'Ausgaben_Süßigkeiten',
       'Ausgaben_Gold', 'Anzahl_Rabattkäufe', 'Anzahl_Webkäufe',
       'Anzahl_Katalogkäufe', 'Anzahl_Ladeneinkäufe',
       'Anzahl_WebBesuche_Monat', 'Alter']]

In [ ]:
sns.pairplot(df_scatter, hue='Familienstand', diag_kind='kde',palette="tab10")
plt.show()

In [ ]:
kampagnen_spalten = ["Kampagne_1_Akzeptiert","Kampagne_2_Akzeptiert",
                     "Kampagne_3_Akzeptiert","Kampagne_4_Akzeptiert","Kampagne_5_Akzeptiert"]

kampagnen_counts = df[kampagnen_spalten].sum()

plt.figure(figsize=(12,5))
sns.barplot(x=kampagnen_counts.index, y=kampagnen_counts.values, palette='Set2')
plt.title("Akzeptanz pro Kampagne")
plt.ylabel("Anzahl akzeptiert")
plt.show()

In [ ]:
kanal_spalten = ["Anzahl_Webkäufe","Anzahl_Katalogkäufe","Anzahl_Ladeneinkäufe"]

kanal_counts = df[kanal_spalten].sum()

plt.figure(figsize=(8,5))
sns.barplot(x=kanal_counts.index, y=kanal_counts.values,palette='Set2')
plt.title("Käufe nach Kanal")
plt.ylabel("Gesamtanzahl")
plt.show()

In [ ]:
web_spalten = ["Anzahl_Webkäufe", 'Anzahl_WebBesuche_Monat']

web_counts = df[web_spalten].sum()

plt.figure(figsize=(8,5))
sns.barplot(x=web_counts.index, y=web_counts.values,palette='Set2')
plt.title("Webkäufe vs Anzahl_WebBesuche")
plt.ylabel("Gesamtanzahl")
plt.show()

In [ ]:
numerische_spalten = [
    "Einkommen",
    "Ausgaben_Wein", "Ausgaben_Obst", "Ausgaben_Fleisch",
    "Ausgaben_Fisch", "Ausgaben_Süßigkeiten", "Ausgaben_Gold",
    "Anzahl_Rabattkäufe", "Anzahl_Webkäufe", "Anzahl_Katalogkäufe",
    "Anzahl_Ladeneinkäufe"
]

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(15,10))  # 3 Reihen, 4 Spalten
axes = axes.flatten()

for i, col in enumerate(numerische_spalten):
    sns.boxplot(y=df[col], ax=axes[i],palette='Set2')
    axes[i].set_title(col)

plt.tight_layout()
plt.show()

# 4. Neue Features erstellen
- Gesamt-Ausgaben = Summe aller Ausgaben-Spalten.
- Gesamt-Kinder = Kinder_zu_Hause + Teenager_zu_Hause.
- Kundenbindungsdauer = Heute − Datum_Kunde.


In [ ]:
df['Gesamt_Ausgaben'] = (
    df['Ausgaben_Wein'] +
    df['Ausgaben_Obst'] +
    df['Ausgaben_Fleisch'] +
    df['Ausgaben_Fisch'] +
    df['Ausgaben_Süßigkeiten'] +
    df['Ausgaben_Gold']
)


In [ ]:
df_einkommen_fehlt['Gesamt_Ausgaben'] = (
    df['Ausgaben_Wein'] +
    df['Ausgaben_Obst'] +
    df['Ausgaben_Fleisch'] +
    df['Ausgaben_Fisch'] +
    df['Ausgaben_Süßigkeiten'] +
    df['Ausgaben_Gold']
)

In [ ]:
df['Gesamt_Kinder'] = df['Kinder_zu_Hause'] + df['Teenager_zu_Hause']
df_einkommen_fehlt['Gesamt_Kinder'] = df_einkommen_fehlt['Kinder_zu_Hause'] + df_einkommen_fehlt['Teenager_zu_Hause']

In [ ]:
df['Kundenbindungsdauer'] =pd.Timestamp.today() - df['Datum_Kunde']
df['Kundenbindungsdauer_Jahre'] = df['Kundenbindungsdauer'].dt.days / 365.25

In [ ]:
df_einkommen_fehlt['Kundenbindungsdauer'] =pd.Timestamp.today() - df_einkommen_fehlt['Datum_Kunde']
df_einkommen_fehlt['Kundenbindungsdauer_Jahre'] = df_einkommen_fehlt['Kundenbindungsdauer'].dt.days / 365.25

In [ ]:
df['Ausgaben_pro_Jahr'] = df['Gesamt_Ausgaben'] / df['Kundenbindungsdauer_Jahre']

In [ ]:
df_einkommen_fehlt['Ausgaben_pro_Jahr'] = df_einkommen_fehlt['Gesamt_Ausgaben'] / df_einkommen_fehlt['Kundenbindungsdauer_Jahre']

In [ ]:
# Verhältnis berechnen
df['Wein_Anteil'] = df['Ausgaben_Wein'] / df['Gesamt_Ausgaben']

# Optional: in Prozent
df['Wein_Anteil_Prozent'] = (df['Wein_Anteil'] * 100).round(2)

In [ ]:
# Verhältnis berechnen
df_einkommen_fehlt['Wein_Anteil'] = df_einkommen_fehlt['Ausgaben_Wein'] / df_einkommen_fehlt['Gesamt_Ausgaben']

# Optional: in Prozent
df_einkommen_fehlt['Wein_Anteil_Prozent'] = (df_einkommen_fehlt['Wein_Anteil'] * 100).round(2)

In [ ]:
df.columns

In [ ]:
df_einkommen_fehlt.columns

In [ ]:
df = df [['ID', 
       'Geburtsjahr','Alter',  'Einkommen','Bildungsniveau', 'Familienstand',
       'Kinder_zu_Hause', 'Teenager_zu_Hause', 'Gesamt_Kinder',
       'Datum_Kunde', 'Kundenbindungsdauer','Letzter_Kauf_Tage', 'Kundenbindungsdauer_Jahre',
       'Ausgaben_Wein','Ausgaben_Fleisch', 'Ausgaben_Obst', 'Ausgaben_Fisch', 'Ausgaben_Süßigkeiten', 'Ausgaben_Gold',  'Ausgaben_pro_Jahr','Gesamt_Ausgaben', 'Wein_Anteil', 'Wein_Anteil_Prozent',
       'Anzahl_Rabattkäufe',  'Anzahl_Katalogkäufe', 'Anzahl_Ladeneinkäufe', 'Anzahl_Webkäufe','Anzahl_WebBesuche_Monat', 
       'Kampagne_1_Akzeptiert', 'Kampagne_2_Akzeptiert',  'Kampagne_3_Akzeptiert','Kampagne_4_Akzeptiert', 'Kampagne_5_Akzeptiert', 'Antwort_Letzte_Kampagne',
       'Beschwerde'
        ]]
df.head(5)

In [ ]:
df.to_csv('Marktkampagne_V1.csv', index=False)

# 5. Lineare Regression Einkommen als ziel variable

In [ ]:
df_Korrelation = df [['Einkommen',
       'Letzter_Kauf_Tage',  'Ausgaben_Wein', 'Ausgaben_Obst',
       'Ausgaben_Fleisch', 'Ausgaben_Fisch', 'Ausgaben_Süßigkeiten',
       'Ausgaben_Gold', 'Anzahl_Rabattkäufe', 'Anzahl_Webkäufe',
       'Anzahl_Katalogkäufe', 'Anzahl_Ladeneinkäufe',
       'Anzahl_WebBesuche_Monat', 'Alter',  'Gesamt_Kinder',
       'Kundenbindungsdauer']]

In [ ]:
# Korrelationsmatrix nur für kontinuierliche Features
korrelation2 = df_Korrelation.corr()

# Heatmap

plt.figure(figsize=(12,10))
sns.heatmap(korrelation2, annot=True, fmt=".2f", cmap='coolwarm')
plt.title("Korrelationsmatrix der kontinuierlichen numerischen Features")
plt.show()

Merkmale mit sehr geringer Korrelation und nahezu null Koeffizienten wurden entfernt,  
da sie keinen signifikanten Beitrag zur Erklärung der Zielvariable leisten und das Modell unnötig verkomplizieren.


In [ ]:
df_Korrelation2 = df [['Einkommen' , 'Alter' , 'Gesamt_Kinder',
        'Ausgaben_Wein', 'Ausgaben_Obst', 'Ausgaben_Fleisch', 'Ausgaben_Fisch', 'Ausgaben_Süßigkeiten','Ausgaben_Gold', 
        'Anzahl_Rabattkäufe','Anzahl_Katalogkäufe', 'Anzahl_Ladeneinkäufe',  'Anzahl_Webkäufe' ,'Anzahl_WebBesuche_Monat', 'Kundenbindungsdauer']]

In [ ]:
# Korrelation (Pearson) berechnen
korrelation2 = df_Korrelation2.corr()

# Absolute Korrelation
korrelation_abs = korrelation2.abs()


# Heatmap

plt.figure(figsize=(12,10))
sns.heatmap(korrelation_abs, annot=True, fmt=".2f", cmap='coolwarm')
plt.title("Korrelationsmatrix der kontinuierlichen numerischen Features")
plt.show()

- Es wird ein DataFrame `df_L_R` erstellt, das alle für die lineare Regression relevanten Merkmale enthält.

In [ ]:
df_L_R = df  [['Einkommen' , 'Alter' , 'Bildungsniveau', 'Familienstand', 'Gesamt_Kinder',
        'Ausgaben_Wein', 'Ausgaben_Obst', 'Ausgaben_Fleisch', 'Ausgaben_Fisch', 'Ausgaben_Süßigkeiten','Ausgaben_Gold', 
        'Anzahl_Rabattkäufe','Anzahl_Katalogkäufe', 'Anzahl_Ladeneinkäufe',  'Anzahl_Webkäufe' ,'Anzahl_WebBesuche_Monat', 'Kundenbindungsdauer']]

In [ ]:
df_L_R = df_L_R.copy()  # wichtig, um SettingWithCopyWarning zu vermeiden

df_L_R.loc[:, 'Kundenbindungsdauer_Tage'] = df_L_R['Kundenbindungsdauer'].dt.days 


In [ ]:
df_L_R = df_L_R.drop(columns=['Kundenbindungsdauer'])
df_L_R.rename(columns={'Kundenbindungsdauer_Tage': 'Kundenbindungsdauer'}, inplace=True)


In [ ]:
df_L_R.info()

In [ ]:
df_L_R.head(5)

In [ ]:
df_L_R.columns

### Merkmale und Zielvariable ("Features" und "target variable")


In [ ]:
numerical_features = [ 'Alter', 'Gesamt_Kinder', 
       'Ausgaben_Wein', 'Ausgaben_Obst', 'Ausgaben_Fleisch','Ausgaben_Fisch', 'Ausgaben_Süßigkeiten', 'Ausgaben_Gold',
       'Anzahl_Rabattkäufe', 'Anzahl_Katalogkäufe', 'Anzahl_Ladeneinkäufe','Anzahl_Webkäufe', 'Anzahl_WebBesuche_Monat', 'Kundenbindungsdauer']

categorical_features = ['Bildungsniveau', 'Familienstand']

features = numerical_features + categorical_features

target_variable = 'Einkommen'

In [ ]:
X = df_L_R[features]

In [ ]:
y = df_L_R[target_variable]

### Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Modell-Pipeline


#### LinearRegression

In [ ]:

# Spaltentransformation mit ColumnTransformer
transformer = ColumnTransformer([
    ('scaling', StandardScaler(), numerical_features), 
    ('onehot', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
])



In [ ]:
# Pipeline erstellen
pipeline = Pipeline([
    ('col_transformer', transformer),
    ('lr_model', LinearRegression())
])
# Pipeline trainieren
pipeline.fit(X_train, y_train)

**Vorhersagen**

In [ ]:
# Vorhersage auf Trainingsdaten
y_pred_train = pipeline.predict(X_train)
# Vorhersage auf Testdaten
y_pred_test = pipeline.predict(X_test)

In [ ]:
plt.scatter(y_train, y_pred_train, color='blue', s=8)
plt.scatter(y_test, y_pred_test, color='red', s=8)
plt.plot([2000, 7000], [2000, 7000], color='green')
plt.xlabel('Tatsächliche Werte')
plt.ylabel('Vorhergesagte Werte')

##### Maß zur Beurteilung des Ergebnisses: R2

In [ ]:
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)

print(f'R² Trainingsdaten: {r2_train:.4f}')
print(f'R² Testdaten: {r2_test:.4f}')

Modell erklärt ≈ 75 % der Varianz des Einkommens durch die gewählten Merkmale.
Train ≈ Test → kein Overfitting

Test sogar minimal besser → normal durch Stichprobenrauschen

**Model Pipeline mit Polynom-Termen**

Wir haben eine Polynomial Regression getestet, diese ist in diesem Fall jedoch **nicht sinnvoll**.

- R² Trainingsdaten: **0,8432**
- R² Testdaten: **0,5929**

Der deutliche Unterschied zwischen Trainings- und Testleistung zeigt, dass das Modell **overfittet**:
Es passt sich zu stark an die Trainingsdaten an und generalisiert schlecht auf neue, unbekannte Daten.


In [ ]:
# # (kleine) Numerische Pipeline
# numerical_pipeline = Pipeline([
#     ('scaling', StandardScaler()),
#     ('polynome', PolynomialFeatures(degree=2, include_bias=False))
# ])


# # Spaltentransformation mit ColumnTransformer
# transformer = ColumnTransformer([
#     ('numerical_pl', numerical_pipeline, numerical_features), 
#     ('onehot', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
# ])

# # Pipeline erstellen
# pipeline = Pipeline([
#     ('col_transformer', transformer),
#     ('lr_model', LinearRegression())
# ])



Modell erklärt ≈ 75 % der Varianz des Einkommens durch die gewählten Merkmale.
Train ≈ Test → kein Overfitting

Test sogar minimal besser → normal durch Stichprobenrauschen

In [ ]:
# Vorhersage auf Testdaten
y_pred = pipeline.predict(X_test)

# Residuen
residuen = y_test - y_pred

print(residuen.describe())



In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(y_pred, residuen)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Vorhergesagtes Einkommen')
plt.ylabel('Residuen')
plt.title('Residuen vs. Vorhersage')
plt.show()

In [ ]:
sns.histplot(residuen, kde=True)
plt.title('Verteilung der Residuen')
plt.show()

🔹 Quartile

25% = −6.254 → unteres Quartil leicht negativ

50% = −0.131 → Median sehr nah bei 0 ✅

75% = 6.127 → obere Hälfte leicht positiv

➡️ Residuen sind symmetrisch um 0 verteilt → Modell linear sinnvoll

Funktion erstellen um mehrere Model zu teste

In [ ]:
# Spaltentransformation mit ColumnTransformer
transformer = ColumnTransformer([
    ('scaling', StandardScaler(), numerical_features), 
    ('onehot', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
])

In [ ]:
def train_and_evaluate(model, model_name, plot=False):
    pipeline = Pipeline([
        ("col_transformer", transformer),
        ("model", model)
    ])

    # Training
    pipeline.fit(X_train, y_train)

    # Vorhersagen
    y_pred_train = pipeline.predict(X_train)
    y_pred_test = pipeline.predict(X_test)

    # Metriken
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    r2_train = r2_score(y_train, y_pred_train)
    r2_test = r2_score(y_test, y_pred_test)

    if plot:
        plt.figure(figsize=(6, 6))
        plt.scatter(y_test, y_pred_test, s=8, alpha=0.6)
        plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color="red")
        plt.xlabel("Tatsächliche Werte")
        plt.ylabel("Vorhersage")
        plt.title(model_name)
        plt.show()

    print(f"\n=== {model_name} ===")
    print(f"RMSE (Test)    = {rmse:.2f}")
    print(f"R² Train       = {r2_train:.3f}")
    print(f"R² Test        = {r2_test:.3f}")

    return pipeline

In [ ]:
# Lineare Regression
lr_pipeline = train_and_evaluate(
    LinearRegression(fit_intercept=True),
    "Linear Regression"
)

# Ridge Regression
ridge_pipeline = train_and_evaluate(
    Ridge(alpha=1.0, fit_intercept=True),
    "Ridge Regression"
)

# Lasso Regression
lasso_pipeline = train_and_evaluate(
    Lasso(alpha=0.001, fit_intercept=True, max_iter=10000),
    "Lasso Regression"
)

In [ ]:
df_einkommen_fehlt.head(5)

In [ ]:
df_einkommen_fehlt['Kundenbindungsdauer'] = df_einkommen_fehlt['Kundenbindungsdauer'].dt.days.astype(int)


In [ ]:
df_einkommen_fehlt.columns

In [ ]:
df_L_R.columns

In [ ]:
X_missing = df_einkommen_fehlt[[ 'Bildungsniveau', 'Familienstand', 
      
       'Ausgaben_Wein', 'Ausgaben_Obst',
       'Ausgaben_Fleisch', 'Ausgaben_Fisch', 'Ausgaben_Süßigkeiten',
       'Ausgaben_Gold', 'Anzahl_Rabattkäufe', 'Anzahl_Webkäufe',
       'Anzahl_Katalogkäufe', 'Anzahl_Ladeneinkäufe',
       'Anzahl_WebBesuche_Monat', 'Alter',  'Gesamt_Kinder','Kundenbindungsdauer']]


In [ ]:
X_missing.info()

In [ ]:
df_einkommen_fehlt['Einkommen'] = pipeline.predict(X_missing)


In [ ]:
df_einkommen_fehlt['Gesamt_Ausgaben'] = (
    df_einkommen_fehlt['Ausgaben_Wein'] +
    df_einkommen_fehlt['Ausgaben_Obst'] +
    df_einkommen_fehlt['Ausgaben_Fleisch'] +
    df_einkommen_fehlt['Ausgaben_Fisch'] +
    df_einkommen_fehlt['Ausgaben_Süßigkeiten'] +
    df_einkommen_fehlt['Ausgaben_Gold']
)
df_einkommen_fehlt['Kundenbindungsdauer'] =pd.Timestamp.today() - df_einkommen_fehlt['Datum_Kunde']
df_einkommen_fehlt['Kundenbindungsdauer_Jahre'] = df_einkommen_fehlt['Kundenbindungsdauer'].dt.days / 365.25
df_einkommen_fehlt['Gesamt_Kinder'] = df_einkommen_fehlt['Kinder_zu_Hause'] + df_einkommen_fehlt['Teenager_zu_Hause']
df_einkommen_fehlt['Ausgaben_pro_Jahr'] = df_einkommen_fehlt['Gesamt_Ausgaben'] / df_einkommen_fehlt['Kundenbindungsdauer_Jahre']
df_einkommen_fehlt['Wein_Anteil'] = df_einkommen_fehlt['Ausgaben_Wein'] / df_einkommen_fehlt['Gesamt_Ausgaben']
df_einkommen_fehlt['Wein_Anteil_Prozent'] = (df_einkommen_fehlt['Wein_Anteil'] * 100).round(2)


In [ ]:
df.columns

In [ ]:
df_einkommen_fehlt = df_einkommen_fehlt[['ID', 'Geburtsjahr', 'Alter', 'Einkommen', 'Bildungsniveau',
       'Familienstand', 'Kinder_zu_Hause', 'Teenager_zu_Hause',
       'Gesamt_Kinder', 'Datum_Kunde', 'Kundenbindungsdauer',
       'Letzter_Kauf_Tage', 'Kundenbindungsdauer_Jahre', 'Ausgaben_Wein',
       'Ausgaben_Fleisch', 'Ausgaben_Obst', 'Ausgaben_Fisch',
       'Ausgaben_Süßigkeiten', 'Ausgaben_Gold', 'Ausgaben_pro_Jahr',
       'Gesamt_Ausgaben', 'Wein_Anteil', 'Wein_Anteil_Prozent',
       'Anzahl_Rabattkäufe', 'Anzahl_Katalogkäufe', 'Anzahl_Ladeneinkäufe',
       'Anzahl_Webkäufe', 'Anzahl_WebBesuche_Monat', 'Kampagne_1_Akzeptiert',
       'Kampagne_2_Akzeptiert', 'Kampagne_3_Akzeptiert',
       'Kampagne_4_Akzeptiert', 'Kampagne_5_Akzeptiert',
       'Antwort_Letzte_Kampagne', 'Beschwerde']]

In [ ]:
df_einkommen_fehlt.isna().sum()

In [ ]:
df = pd.concat([df, df_einkommen_fehlt], ignore_index=True)


In [ ]:
df.to_csv('Marktkampagne_V2.csv', index=False)

# 6. Lenear  Regression ZielVariable Gesamt Aufgabe

In [ ]:
df.info()

In [ ]:
df = df.copy()  # wichtig, um SettingWithCopyWarning zu vermeiden

df.loc[:, 'Kundenbindungsdauer_Tage'] = df['Kundenbindungsdauer'].dt.days 
df = df.drop(columns=['Kundenbindungsdauer'])
df.rename(columns={'Kundenbindungsdauer_Tage': 'Kundenbindungsdauer'}, inplace=True)


In [ ]:
df.info()

In [ ]:
df_L_R_2 = df [['Alter', 'Einkommen', 'Bildungsniveau',
       'Familienstand',
       'Gesamt_Kinder', 'Kundenbindungsdauer',
       'Gesamt_Ausgaben', 'Anzahl_Rabattkäufe',
       'Anzahl_Katalogkäufe', 'Anzahl_Ladeneinkäufe', 'Anzahl_Webkäufe',
       'Anzahl_WebBesuche_Monat'
    
]]

In [ ]:
df_Korrelation3 = df_L_R_2 [['Alter', 'Einkommen', 
       'Gesamt_Kinder', 'Kundenbindungsdauer',
       'Gesamt_Ausgaben', 'Anzahl_Rabattkäufe',
       'Anzahl_Katalogkäufe', 'Anzahl_Ladeneinkäufe', 'Anzahl_Webkäufe',
       'Anzahl_WebBesuche_Monat'
    
]]

In [ ]:
# Korrelationsmatrix nur für kontinuierliche Features
korrelation3 = df_Korrelation3.corr()

# Heatmap

plt.figure(figsize=(12,10))
sns.heatmap(korrelation3, annot=True, fmt=".2f", cmap='coolwarm')
plt.title("Korrelationsmatrix der kontinuierlichen numerischen Features")
plt.show()

In [ ]:
numerical_features = [ 'Alter', 'Einkommen', 
       'Gesamt_Kinder', 'Kundenbindungsdauer',
       'Anzahl_Rabattkäufe',
       'Anzahl_WebBesuche_Monat'
    ]

categorical_features = ['Bildungsniveau', 'Familienstand']

features = numerical_features + categorical_features

target_variable = 'Gesamt_Ausgaben'

In [ ]:
X = df_L_R_2[features]

In [ ]:
y = df_L_R_2[target_variable]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Spaltentransformation mit ColumnTransformer
transformer = ColumnTransformer([
    ('scaling', StandardScaler(), numerical_features), 
    ('onehot', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
])


In [ ]:
# Pipeline erstellen
pipeline = Pipeline([
    ('col_transformer', transformer),
    ('lr_model', LinearRegression())
])
# Pipeline trainieren
pipeline.fit(X_train, y_train)

In [ ]:
# Vorhersage auf Trainingsdaten
y_pred_train = pipeline.predict(X_train)
# Vorhersage auf Testdaten
y_pred_test = pipeline.predict(X_test)

In [ ]:
plt.scatter(y_train, y_pred_train, color='blue', s=8)
plt.scatter(y_test, y_pred_test, color='red', s=8)
plt.plot([2000, 7000], [2000, 7000], color='green')
plt.xlabel('Tatsächliche Werte')
plt.ylabel('Vorhergesagte Werte')

In [ ]:
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)

print(f'R² Trainingsdaten: {r2_train:.4f}')
print(f'R² Testdaten: {r2_test:.4f}')